# 4. Computing the inverse solution

In this notebook we will learn about the inverse model and how it provives the solution to the so-called inverse problem.


<div class="alert alert-success">
    <b>Learning Objectives</b>:
     <ul>
      <li>Understanding how to compute the inverse solution with MNE python </li>
      <li>Exploring two different inverse solutions (MNE, LCMV Beamformer) </li>
    </ul>
</div>

In [ ]:
## TODO: insert picture showing the 

## 3.2 The inverse problem is ill-posed

We know $\mathbf{y}$, we know $\mathbf{L}$, and we want $\mathbf{x}$. We cannot simply invert $\mathbf{L}$, for two independent reasons.

**1. It is underdetermined.** A typical source space has $M \approx 5\,000$–$20\,000$ sources, while $N$ is at most a few hundred channels, so $M \gg N$. The lead field therefore has a large null space: there exist non-zero source patterns $\mathbf{x}_0$ with

$$\mathbf{L}\,\mathbf{x}_{0} = \mathbf{0}$$

These are **silent sources** — activity that produces literally no measurement. Any multiple of $\mathbf{x}_0$ can be added to a solution without changing the data by one microvolt, so infinitely many source configurations explain the same recording *exactly*. No amount of clean data fixes this; it is a property of the physics, not of the noise.

**2. It is ill-conditioned.** Even inside the row space of $\mathbf{L}$, deep and radially oriented sources project onto the sensors very weakly. Those directions correspond to tiny singular values of $\mathbf{L}$, and naively inverting them multiplies the measurement noise by enormous factors.

The consequence: **the inverse problem has no unique solution, so we must add assumptions.** Every method below is a different assumption about which of the infinitely many candidate solutions we prefer. Choosing an inverse method *is* choosing a prior — it is not a neutral technical step.

## 3.3 All the methods here are linear filters

Every method in this notebook estimates the sources with a single matrix $\mathbf{W}$ of shape $M \times N$, applied to the data:

$$\hat{\mathbf{x}}(t) \;=\; \mathbf{W}\,\mathbf{y}(t)$$

$\mathbf{W}$ is the *inverse operator*, and row $k$ of $\mathbf{W}$ is the **spatial filter** for source $k$. The methods differ only in how $\mathbf{W}$ is constructed — which is why MNE separates building the operator from applying it (`make_inverse_operator` / `apply_inverse`, `make_lcmv` / `apply_lcmv`).

### Minimum-norm estimates (MNE, dSPM, sLORETA)

Pick the solution that fits the data while keeping total source power small:

$$\hat{\mathbf{x}} \;=\; \arg\min_{\mathbf{x}}\;
\underbrace{\left\lVert \mathbf{y}-\mathbf{L}\mathbf{x} \right\rVert^{2}_{\mathbf{C}^{-1}}}_{\text{data fit}}
\;+\; \lambda^{2}\,\underbrace{\left\lVert \mathbf{x} \right\rVert^{2}_{\mathbf{R}^{-1}}}_{\text{prior}}$$

which has the closed-form solution

$$\mathbf{W} \;=\; \mathbf{R}\mathbf{L}^{\mathsf{T}}
\left( \mathbf{L}\mathbf{R}\mathbf{L}^{\mathsf{T}} + \lambda^{2}\mathbf{C} \right)^{-1}$$

| symbol | what it is | where it comes from in MNE |
|---|---|---|
| $\mathbf{C}$ | noise covariance — whitens the sensors | `noise_cov`, estimated from the baseline |
| $\mathbf{R}$ | source covariance — the prior | shaped by `depth` and `loose` |
| $\lambda^{2}$ | regularisation strength | `lambda2 = 1.0 / snr ** 2` |

$\mathbf{R} = \mathbf{I}$ is plain MNE: *the smallest total source power that explains the data*. This prior is biased towards superficial sources, which is what `depth=0.8` compensates for. **dSPM** and **sLORETA** keep the same $\mathbf{W}$ but rescale each row — by the projected noise (dSPM) or by the estimated source variance (sLORETA) — turning amplitudes into noise-normalised statistical maps.

### LCMV beamformer

A beamformer builds each spatial filter separately, and uses the **data** covariance $\mathbf{C}_d$ rather than a fixed prior. For source $k$ with lead field $\mathbf{L}_k$:

$$\mathbf{w}_{k} \;=\; \arg\min_{\mathbf{w}}\; \mathbf{w}^{\mathsf{T}}\mathbf{C}_{d}\mathbf{w}
\qquad \text{subject to} \qquad \mathbf{w}^{\mathsf{T}}\mathbf{L}_{k} = 1$$

*Minimise total output power, but keep unit gain at the source of interest.* Anything the filter passes that is not source $k$ costs variance, so the filter learns to suppress it. The solution is

$$\mathbf{w}_{k} \;=\;
\frac{\mathbf{C}_{d}^{-1}\mathbf{L}_{k}}{\mathbf{L}_{k}^{\mathsf{T}}\mathbf{C}_{d}^{-1}\mathbf{L}_{k}}$$

Because $\mathbf{C}_d$ is estimated from the data and must be inverted, it is regularised as $\mathbf{C}_d + \alpha \,\frac{\operatorname{tr}(\mathbf{C}_d)}{N}\mathbf{I}$ — that is the `reg=0.05` argument. With `weight_norm="unit-noise-gain"` the filter is instead normalised by $\lVert \mathbf{w}_k \rVert$, so that projected noise is constant across the brain and deep sources stop looking artificially quiet.

Two consequences worth carrying into the exercises: a beamformer is **adaptive** (change the time window or the filter band and every spatial filter changes), and it assumes sources are not perfectly correlated — two sources locked in phase partly cancel each other.

## 3.4 What you actually get back

Substituting the forward model into the estimate shows what any linear method really returns:

$$\hat{\mathbf{x}} \;=\; \mathbf{W}\mathbf{y}
\;=\; \underbrace{\mathbf{W}\mathbf{L}}_{\textstyle \mathbf{Res}}\,\mathbf{x} \;+\; \mathbf{W}\mathbf{n}$$

$\mathbf{Res} = \mathbf{W}\mathbf{L}$ is the **resolution matrix**. Perfect reconstruction would mean $\mathbf{Res} = \mathbf{I}$; it never is, for any method, because of the null space in §3.2.

- **Column $k$** — the *point-spread function*: how a single source at $k$ smears across your map.
- **Row $k$** — the *cross-talk function*: how much activity from everywhere else leaks into the time course you extract at $k$.

This is source leakage, and it is the reason a parcel time course is never "the activity of that parcel". Keep $\mathbf{Res}$ in mind before interpreting any connectivity or ROI result.

<div class="alert alert-warning">
    <b>Exercise</b>:
     <ul>
      <li>... </li>
    </ul>
</div>

## 3.5 ... Lets get Hands-on!
1. Load pre-computed the forward solution for this head model and source space — the lead field $\mathbf{L}$ of §3.1.
2. Load the epochs and the noise and data covariances computed in notebook 02.
3. Build $\mathbf{W}$ with `make_inverse_operator` and apply minimum norm estimate.
4. Build $\mathbf{W}$ with `make_lcmv` and apply the beamformer.
5. Compare the two on the same data — and look at what each one does to a known source.

In [ ]:
# imports
import os
import mne
import numpy as np
import matplotlib.pyplot as plt

from mne.minimum_norm import make_inverse_operator, apply_inverse

# data paths - everything for this participant shares one prefix
subID = 24
data_path = "data"
subject_path = os.path.join(f"sub-0{subID}", "ses-mecha", "eeg")
base = os.path.join("..", data_path, subject_path, f"sub-0{subID}_ses-mecha_task-NT_")
print(base)

### Load the forward model

The forward solution was computed in notebook 03 from the BEM head model, the source space
and the electrode positions. It contains the lead field $\mathbf{L}$ of §3.1: one scalp
topography per candidate source.

In [ ]:
# load forward model
fwd = mne.read_forward_solution(base + "fwd.fif")
print(fwd)

# the lead field itself: (n_channels, n_sources)
print("lead field shape:", fwd["sol"]["data"].shape)
print("source orientation:", fwd["source_ori"])

This forward solution was saved with **fixed orientations**: each source point carries a
single dipole, clamped perpendicular to the cortical surface. So $\mathbf{L}$ is
$N \times M$ and not $N \times 3M$, and the source estimates we get out will be *signed* -
positive means current flowing outward along the surface normal, negative inward.

### Load the epochs

The same epochs as in notebook 02, plus the two conditions we want to compare.

In [ ]:
# load epochs
epochs = mne.read_epochs(base + "epo.fif", preload=True)

evoked = epochs.average()
evoked_perceived = epochs["response_decoded == 'Yes'"].average()
evoked_unperceived = epochs["response_decoded == 'No'"].average()

print(evoked)

### Noise covariance

The noise covariance $\mathbf{C}$ whitens the sensors: it tells the inverse operator how
noisy each channel is and how the noise is correlated across channels. We estimate it from
the pre-stimulus baseline, where by assumption there is no evoked activity.

`method=["shrunk", "empirical"]` lets MNE fit both estimators and pick the one with the best
cross-validated log-likelihood. `rank="info"` makes it respect the rank deficiency caused by
the average reference (62 channels, rank 61).

In [ ]:
# compute noise covariance from the baseline
noise_cov = mne.compute_covariance(epochs, tmax=0.0, method=["shrunk", "empirical"],
                                   rank="info")
print(noise_cov)
print("selected method:", noise_cov["method"])

In [ ]:
# the covariance matrix and its eigenvalue spectrum
mne.viz.plot_cov(noise_cov, epochs.info)

In [ ]:
# does it whiten the data? after whitening, the baseline should sit inside +-2
# and the channels should look like unit-variance noise
evoked.plot_white(noise_cov)

### Build the inverse operator

`make_inverse_operator` assembles $\mathbf{W}$ from the lead field, the noise covariance and
the source prior $\mathbf{R}$.

Two arguments are forced by our forward solution:

- `loose=0.` - the forward is already fixed-orientation, so the inverse must be too. Any
  other value raises *"Forward operator has fixed orientation and can only be used to make a
  fixed-orientation inverse operator"*.
- `depth=None` - depth weighting reshapes $\mathbf{R}$ using the free-orientation lead
  fields, which are no longer in the file. With a fixed forward, MNE refuses `depth`.
  Without it, the solution keeps the superficial bias mentioned in §3.3: keep that in mind
  when reading depth of activation off the maps.

In [ ]:
# compute inverse operator
inverse_operator = make_inverse_operator(evoked.info, fwd, noise_cov,
                                         loose=0.0, depth=None)
print(inverse_operator)

### Apply it to the data

$\hat{\mathbf{x}}(t) = \mathbf{W}\,\mathbf{y}(t)$ - one matrix multiplication per time
point. The regularisation $\lambda^2$ is set from an assumed SNR: `lambda2 = 1 / snr**2`.
A larger $\lambda^2$ trusts the prior more (smoother, weaker estimates), a smaller one
trusts the data more (noisier estimates).

`method="MNE"` gives the plain minimum-norm estimate in ampere-metres. `"dSPM"` and
`"sLORETA"` would return the same map noise-normalised.

In [ ]:
# apply inverse method
snr = 3.0
lambda2 = 1.0 / snr ** 2
method = "MNE"

stc = apply_inverse(evoked, inverse_operator, lambda2, method=method, pick_ori=None)
stc_perceived = apply_inverse(evoked_perceived, inverse_operator, lambda2, method=method)
stc_unperceived = apply_inverse(evoked_unperceived, inverse_operator, lambda2, method=method)

# the contrast: source estimates can be subtracted like arrays
stc_difference = stc_perceived - stc_unperceived

print(stc)

In [ ]:
# where and when is the estimate largest?
vertex, latency = stc.get_peak(tmin=0.05, tmax=0.4)
print(f"peak at vertex {vertex}, {latency * 1000:.0f} ms")

In [ ]:
# a first look: the source time courses of all vertices
plt.plot(stc.times, stc.data[::100, :].T, linewidth=0.5)
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("time (s)")
plt.ylabel("amplitude (Am)")
plt.title("every 100th source")

### Save the source estimates

`stc.save()` writes one file per hemisphere (`...-lh.stc` and `...-rh.stc`). These are still
in the **individual** source space of this participant - notebook 05 morphs them to
*fsaverage* for plotting.

In [ ]:
for name, estimate in [("all", stc),
                       ("perceived", stc_perceived),
                       ("unperceived", stc_unperceived),
                       ("difference", stc_difference)]:
    fname = base + f"MNE-{name}"
    estimate.save(fname, overwrite=True)
    print("saved", fname + "-lh.stc", "/", fname + "-rh.stc")